---

## **Curso**: Residência em Inteligência Artificial  
## **Unidade Curricular**: Visão Computacional  
## **Professor**: Matheus Vanzan  
## **Alunos**: Ana Carolina Dutra, Camila Gois de Jesus e Guilherme Magalhães Junior
## Desafio Semana 4
---

# Desafio Semana 4 - Car Tracking com YOLO

## **1. Contexto**

Uma empresa deseja monitorar o fluxo de veículos em uma avenida usando vídeos de câmeras comuns. A solução deve identificar veículos, acompanhar cada objeto ao longo dos frames e contar quantos veículos cruzam uma linha virtual.

Nas aulas anteriores, você viu:

- detecção de objetos com YOLO
- bounding boxes
- uso de modelos pré-treinados
- tracking de objetos em vídeo
- métricas e análise visual dos resultados

Neste desafio, você irá aplicar esses conceitos em um problema prático de **Car Tracking**.

## **2. Objetivo**

Implementar uma solução de contagem automática de veículos em vídeo usando YOLO.

Você deverá:

1. Carregar um vídeo de tráfego.
2. Executar o tracking dos objetos usando `model.track()` da biblioteca Ultralytics.
3. Usar os IDs gerados pelo tracker para acompanhar cada veículo ao longo do vídeo.
4. Filtrar apenas classes relacionadas a veículos.
5. Criar uma linha virtual de contagem no vídeo.
6. Implementar uma lógica própria para contar veículos que cruzam a linha.
7. Gerar um vídeo final anotado com bounding boxes, IDs, linha virtual e contadores.
8. Apresentar uma breve análise dos resultados.

## **3. Vídeo**

O vídeo pode ser obtido de qualquer fonte publica na internet. Teste a sua solução com, pelo menos, 3 videos diferentes.

## **4. Requisitos de implementação**

### **4.1. Ambiente**

O notebook deve:

- instalar ou importar a biblioteca `ultralytics`
- verificar se a GPU está disponível
- importar `cv2`, `os`, `YOLO` e bibliotecas auxiliares
- configurar a variável `PATH`
- montar o caminho completo do vídeo em `video_path`

### **4.2. Modelo YOLO**

Use um modelo YOLO pré-treinado, por exemplo:

```python
model = YOLO('yolov8s.pt')
```

### **4.3. Tracking**

A etapa principal deve usar `track()`, não apenas `predict()`.

Exemplo esperado de chamada:

```python
results = model.track(
    video_path,
    stream=True
)
```

O uso de `stream=True` é importante para processar o vídeo frame a frame sem carregar todos os resultados de uma vez na memória.

Em cada `result`, utilize as informações retornadas pelo YOLO, como:

```python
result.orig_img
result.boxes.xyxy
result.boxes.cls
result.boxes.id
```

O campo `result.boxes.id` deve ser usado para identificar o mesmo objeto em frames diferentes.

### **4.4. Classes de veículos**

Filtre apenas classes relevantes para contagem de veículos.

No dataset COCO, algumas classes úteis são:

```python
vehicle_classes = [2, 3, 5, 7]
```

Essas classes representam, de forma geral:

- carro
- motocicleta
- ônibus
- caminhão

### **4.5. Linha virtual**

Crie uma linha virtual no vídeo para servir como referência de contagem.

Você pode usar uma linha no meio da imagem ou ajustar a posição conforme o vídeo.

### **4.6. Contagem**

Implemente uma lógica para contar veículos quando eles cruzarem a linha virtual.

A solução deve usar o ID do objeto para evitar contar o mesmo veículo repetidas vezes.

A estratégia exata de contagem fica a seu critério. O importante é justificar, no final, como você decidiu identificar o cruzamento da linha.

### **4.7. Vídeo final**

O vídeo final deve mostrar:

- bounding box dos veículos detectados
- ID do objeto rastreado
- centro do objeto ou algum marcador equivalente
- linha virtual de contagem
- contadores atualizados ao longo do vídeo

## **5. Entregáveis**

Entregue o notebook executado contendo:

- código completo da solução
- vídeo original ou referência ao arquivo usado
- vídeo final anotado
- contagem final de veículos
- breve análise dos resultados

## **6. Análise final**

Ao final, responda em Markdown:

1. A contagem pareceu coerente visualmente?
2. O tracking manteve IDs consistentes ao longo do vídeo?
3. Em quais situações ocorreram erros ou instabilidades?
4. Como você melhoraria a solução?


#1-**Importações**

In [ ]:
!pip install -q ultralytics opencv-python

In [ ]:
import torch

print(torch.cuda.is_available())          # True se tem GPU
print(torch.cuda.device_count())          # Quantas GPUs
print(torch.cuda.get_device_name(0))      # Nome da GPU (se existir)

In [ ]:
import ultralytics
ultralytics.checks()

In [ ]:
import os
import cv2
from ultralytics import YOLO
import kagglehub

#**2 -Busca dos datasets e Tratamento**

Por questões de facilidade e para não precisar armazenar os vídeos, vamos pegar alguns do kaggle, utilizando kagglehub.

In [ ]:


# Download latest version
path = kagglehub.dataset_download("shawon10/road-traffic-video-monitoring")

print("Path to dataset files:", path)

In [ ]:
for root, dirs, files in os.walk(path):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
path = kagglehub.dataset_download("vivek603/vehicle-detection-sample-and-output-videos")

for root, dirs, files in os.walk(path):
    for file in files:
        print(os.path.join(root, file))

Utilizamos os paths como entrada das variáveis. Colabs diferentes podem armazenar as variáveis em locais diferentes, é bom ficar de olho.

In [ ]:
Cam_v1 = "/root/.cache/kagglehub/datasets/shawon10/road-traffic-video-monitoring/versions/3/traffic_video.avi"
Cam_v2 = "/root/.cache/kagglehub/datasets/shawon10/road-traffic-video-monitoring/versions/3/road_trafifc.mp4"
Cam_v3 = "/root/.cache/kagglehub/datasets/shawon10/road-traffic-video-monitoring/versions/3/british_highway_traffic.mp4"
Cam_v4 = "/root/.cache/kagglehub/datasets/vivek603/vehicle-detection-sample-and-output-videos/versions/1/Test Video.mp4"
OUT_v1 = "videos_resultado/resultado_contagem_1.avi"
OUT_v2 = "videos_resultado/resultado_contagem_2.mp4"
OUT_v3 = "videos_resultado/resultado_contagem_3.mp4"
OUT_v4 = "videos_resultado/resultado_contagem_4.mp4"

In [ ]:
'''Cam_v1 = "/kaggle/input/road-traffic-video-monitoring/traffic_video.avi"
Cam_v2 = "/kaggle/input/road-traffic-video-monitoring/road_trafifc.mp4"
Cam_v3 = "/kaggle/input/road-traffic-video-monitoring/british_highway_traffic.mp4"
Cam_v4 = "/kaggle/input/vehicle-detection-sample-and-output-videos/Test Video.mp4"
OUT_v1 = "videos_resultado/resultado_contagem_1.avi"
OUT_v2 = "videos_resultado/resultado_contagem_2.mp4"
OUT_v3 = "videos_resultado/resultado_contagem_3.mp4"
OUT_v4 = "videos_resultado/resultado_contagem_4.mp4"'''

Do jeito que estão, os arquivos deram alguns erros de codec, temos que passar v1 e v3 de mpeg4 para h264 para parar de ter erros no colab

In [ ]:
!ffmpeg -i "{Cam_v1}" \
-c:v libx264 \
-c:a aac \
Cam_v1.mp4

In [ ]:
!ffmpeg -i "{Cam_v3}" \
-c:v libx264 \
-c:a aac \
Cam_v3.mp4

In [ ]:
Cam_v1 = "Cam_v1.mp4"
Cam_v3 = "Cam_v3.mp4"

#**3- Visualização**

###VIDEO 1

In [ ]:
from IPython.display import Video

Video(Cam_v1, embed=True)

###VÍDEO2

In [ ]:

Video(Cam_v2, embed=True)

###Vídeo 3

In [ ]:

Video(Cam_v3, embed=True)

###Vídeo 4

In [ ]:

Video(Cam_v4, embed=True)

#**4- Treinamento**

Usamos o modelo 11m, como um modelo médio porém rápido

In [ ]:
model = YOLO("yolo11m.pt")

In [ ]:
model.names

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Rodando em: {device.upper()}")

Classe = [2, 3, 5, 7]
historico = {}
veiculos_contados = set()
vrum = 0

def intercepta_linha(p1, p2, q1, q2):
    def ccw(A, B, C):
        return (C[1] - A[1]) * (B[0] - A[0]) > (B[1] - A[1]) * (C[0] - A[0])
    return ccw(p1, q1, q2) != ccw(p2, q1, q2) and ccw(p1, p2, q1) != ccw(p1, p2, q2)

def processar_video(video_in, video_out, linha_contagem):
    global vrum, historico, veiculos_contados

    vrum = 0
    historico = {}
    veiculos_contados = set()

    cap_meta = cv2.VideoCapture(video_in)
    if not cap_meta.isOpened():
        print(f"Erro ao abrir o vídeo: {video_in}")
        return

    width  = int(cap_meta.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap_meta.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = int(cap_meta.get(cv2.CAP_PROP_FPS))
    cap_meta.release()

    os.makedirs(os.path.dirname(video_out), exist_ok=True)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(video_out, fourcc, fps, (width, height))

    pt1, pt2 = linha_contagem
    print(f"Iniciando Tracking do vídeo: {video_in}")

    results_generator = model.track(
        source=video_in,
        stream=True,
        persist=True,
        verbose=False,
        device=device
    )

    for result in results_generator:
        frame = result.orig_img.copy()

        cv2.line(frame, pt1, pt2, (255, 0, 0), 3)
        cv2.putText(frame, "LINHA VIRTUAL", (pt1[0], pt1[1] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

        if result.boxes is not None and result.boxes.id is not None:
            boxes = result.boxes.xyxy.cpu().numpy()
            ids_detectados = result.boxes.id.cpu().numpy().astype(int)
            clss = result.boxes.cls.cpu().numpy().astype(int)

            for box, id_obj, cls in zip(boxes, ids_detectados, clss):
                if cls in Classe:
                    x1, y1, x2, y2 = box

                    cx = int((x1 + x2) / 2)
                    cy = int((y1 + y2) / 2)
                    ponto_atual = (cx, cy)

                    label = f"{model.names[cls]} ID: {id_obj}"
                    cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
                    cv2.circle(frame, ponto_atual, 5, (0, 0, 255), -1)
                    cv2.putText(frame, label, (int(x1), int(y1) - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

                    if id_obj in historico:
                        ponto_anterior = historico[id_obj]

                        if intercepta_linha(pt1, pt2, ponto_anterior, ponto_atual):
                            if id_obj not in veiculos_contados: # <-- Checando no set global correto
                                veiculos_contados.add(id_obj)   # <-- Adicionando no set global correto
                                vrum += 1

                    historico[id_obj] = ponto_atual

        cv2.rectangle(frame, (15, 15), (260, 65), (0, 0, 0), -1)
        cv2.putText(frame, f"CONTAGEM: {vrum}", (25, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

        out.write(frame)

    out.release()
    print(f"Concluído: {video_in} | Total Contado: {vrum}\n")


LINHA_V1 = [(349, 386), (979, 388)]
LINHA_V2 = [(100, 218), (411, 209)]
LINHA_V3 = [(518, 238), (1260, 333)]
LINHA_V4 = [(271, 551), (727, 545)]

processar_video(Cam_v1, OUT_v1, LINHA_V1)
processar_video(Cam_v2, OUT_v2, LINHA_V2)
processar_video(Cam_v3, OUT_v3, LINHA_V3)
processar_video(Cam_v4, OUT_v4, LINHA_V4)

In [ ]:
!ffmpeg -i {OUT_v1} -c:v libx264 -c:a aac -pix_fmt yuv420p -y resultado_v1_web.mp4

#**5-Resultados**

In [ ]:
Video("resultado_v1_web.mp4", embed=True, width=1000)

In [ ]:
!ffmpeg -i {OUT_v2} -c:v libx264 -c:a aac -pix_fmt yuv420p -y resultado_v2_web.mp4

In [ ]:
Video("resultado_v2_web.mp4", embed=True, width=1000)

In [ ]:
!ffmpeg -i {OUT_v3} -c:v libx264 -c:a aac -pix_fmt yuv420p -y resultado_v3_web.mp4

In [ ]:
Video("resultado_v3_web.mp4", embed=True, width=1000)

In [ ]:
!ffmpeg -i {OUT_v4} -c:v libx264 -c:a aac -pix_fmt yuv420p -y resultado_v4_web.mp4

In [ ]:
Video("resultado_v4_web.mp4", embed=True, width=1000)

#**6-Conclusão**

As contagens, no geral, parecem corretas visualmente, exceto pelo vídeo 2, onde o ângulo da câmera e da linha realmente não ajudaram muito.

Os IDs se mantinham consistentes, exceto quando havia alguma confusão de classe. No entanto, não pareceu, ao nosso ver, ter afetado as contagens.


Erros e instabilidades aconteciam variando entre as distâncias e quando havia alguma confusão de classes, tal como "truck" e "car". Geralmente aconteciam quando havia uma van envolvida.

Para esse exercício, seria interessante, observar opções de fine-tuning, com datasets como o bdd100k e o visdrone

#EXTRAS

###Bytetrack


o YOLO tem outra alternativa para tracking: o bytetrack. Vamos testar!

In [ ]:
OUT_v1 = "Track/resultado_contagem_1.avi"
OUT_v2 = "Track/resultado_contagem_2.mp4"
OUT_v3 = "Track/resultado_contagem_3.mp4"
OUT_v4 = "Track/resultado_contagem_4.mp4"

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Rodando em: {device.upper()}")

Classe = [2, 3, 5, 7]
historico = {}
veiculos_contados = set()
vrum = 0

def intercepta_linha(p1, p2, q1, q2):
    def ccw(A, B, C):
        return (C[1] - A[1]) * (B[0] - A[0]) > (B[1] - A[1]) * (C[0] - A[0])
    return ccw(p1, q1, q2) != ccw(p2, q1, q2) and ccw(p1, p2, q1) != ccw(p1, p2, q2)

def processar_video(video_in, video_out, linha_contagem):
    global vrum, historico, veiculos_contados

    vrum = 0
    historico = {}
    veiculos_contados = set()

    cap_meta = cv2.VideoCapture(video_in)
    if not cap_meta.isOpened():
        print(f"Erro ao abrir o vídeo: {video_in}")
        return

    width  = int(cap_meta.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap_meta.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = int(cap_meta.get(cv2.CAP_PROP_FPS))
    cap_meta.release()

    os.makedirs(os.path.dirname(video_out), exist_ok=True)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(video_out, fourcc, fps, (width, height))

    pt1, pt2 = linha_contagem
    print(f"Iniciando Tracking do vídeo: {video_in}")

    results_generator = model.track(
        source=video_in,
        stream=True,
        persist=True,
        verbose=False,
        device=device,
        tracker="bytetrack.yaml"
    )

    for result in results_generator:
        frame = result.orig_img.copy()

        cv2.line(frame, pt1, pt2, (255, 0, 0), 3)
        cv2.putText(frame, "LINHA VIRTUAL", (pt1[0], pt1[1] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

        if result.boxes is not None and result.boxes.id is not None:
            boxes = result.boxes.xyxy.cpu().numpy()
            ids_detectados = result.boxes.id.cpu().numpy().astype(int)
            clss = result.boxes.cls.cpu().numpy().astype(int)

            for box, id_obj, cls in zip(boxes, ids_detectados, clss):
                if cls in Classe:
                    x1, y1, x2, y2 = box

                    cx = int((x1 + x2) / 2)
                    cy = int((y1 + y2) / 2)
                    ponto_atual = (cx, cy)

                    label = f"{model.names[cls]} ID: {id_obj}"
                    cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
                    cv2.circle(frame, ponto_atual, 5, (0, 0, 255), -1)
                    cv2.putText(frame, label, (int(x1), int(y1) - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

                    if id_obj in historico:
                        ponto_anterior = historico[id_obj]

                        if intercepta_linha(pt1, pt2, ponto_anterior, ponto_atual):
                            if id_obj not in veiculos_contados: # <-- Checando no set global correto
                                veiculos_contados.add(id_obj)   # <-- Adicionando no set global correto
                                vrum += 1

                    historico[id_obj] = ponto_atual

        cv2.rectangle(frame, (15, 15), (260, 65), (0, 0, 0), -1)
        cv2.putText(frame, f"CONTAGEM: {vrum}", (25, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

        out.write(frame)

    out.release()
    print(f"Concluído: {video_in} | Total Contado: {vrum}\n")


LINHA_V1 = [(349, 386), (979, 388)]
LINHA_V2 = [(100, 218), (411, 209)]
LINHA_V3 = [(518, 238), (1260, 333)]
LINHA_V4 = [(271, 551), (727, 545)]

processar_video(Cam_v1, OUT_v1, LINHA_V1)
processar_video(Cam_v2, OUT_v2, LINHA_V2)
processar_video(Cam_v3, OUT_v3, LINHA_V3)
processar_video(Cam_v4, OUT_v4, LINHA_V4)

In [ ]:
!ffmpeg -i {OUT_v1} -c:v libx264 -c:a aac -pix_fmt yuv420p -y resultado_v1_web.mp4

In [ ]:
Video("resultado_v1_web.mp4", embed=True, width=1000)

In [ ]:
!ffmpeg -i {OUT_v2} -c:v libx264 -c:a aac -pix_fmt yuv420p -y resultado_v2_web.mp4

In [ ]:
Video("resultado_v2_web.mp4", embed=True, width=1000)

In [ ]:
!ffmpeg -i {OUT_v3} -c:v libx264 -c:a aac -pix_fmt yuv420p -y resultado_v3_web.mp4

In [ ]:
Video("resultado_v3_web.mp4", embed=True, width=1000)

In [ ]:
!ffmpeg -i {OUT_v4} -c:v libx264 -c:a aac -pix_fmt yuv420p -y resultado_v4_web.mp4

In [ ]:
Video("resultado_v4_web.mp4", embed=True, width=1000)

####Conclusão

Sinceramente, apesar de parecer detectar mais a fundo e ser mais estável para os IDs, para a atividade no geral, não aparentou ter efeitos muito fortes. Ainda sim, talvez para um projeto maior, ela se sairia melhor.

###TESTE

In [ ]:
OUT_v1 = "teste/resultado_contagem_1.avi"
OUT_v2 = "teste/resultado_contagem_2.mp4"
OUT_v3 = "teste/resultado_contagem_3.mp4"
OUT_v4 = "teste/resultado_contagem_4.mp4"

In [ ]:
#Retiramos o finetunning do notebook, dado que consumia muita ram. Foram 30 epocas. YOLO11n -> visdrone


# Carregando seu modelo customizado do VisDrone
model = YOLO("/content/best (1).pt")

print(f" Rodando em: {device.upper()}")

#  Classes de veículos no dataset VisDrone
# 3: carro, 4: van, 5: caminhão, 8: onibus, 9:moto
Classe = [3, 4, 5, 8, 9]

historico = {}
veiculos_contados = set()
vrum = 0

def intercepta_linha(p1, p2, q1, q2):
    def ccw(A, B, C):
        return (C[1] - A[1]) * (B[0] - A[0]) > (B[1] - A[1]) * (C[0] - A[0])
    return ccw(p1, q1, q2) != ccw(p2, q1, q2) and ccw(p1, p2, q1) != ccw(p1, p2, q2)

def processar_video(video_in, video_out, linha_contagem):
    global vrum, historico, veiculos_contados

    vrum = 0
    historico = {}
    veiculos_contados = set()

    cap_meta = cv2.VideoCapture(video_in)
    if not cap_meta.isOpened():
        print(f"Erro ao abrir o vídeo: {video_in}")
        return

    width  = int(cap_meta.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap_meta.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = int(cap_meta.get(cv2.CAP_PROP_FPS))
    cap_meta.release()

    os.makedirs(os.path.dirname(video_out), exist_ok=True)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(video_out, fourcc, fps, (width, height))

    pt1, pt2 = linha_contagem
    print(f"Iniciando Tracking do vídeo: {video_in}")

    results_generator = model.track(
        source=video_in,
        stream=True,
        persist=True,
        verbose=False,
        device=device
    )

    for result in results_generator:
        frame = result.orig_img.copy()

        cv2.line(frame, pt1, pt2, (255, 0, 0), 3)
        cv2.putText(frame, "LINHA VIRTUAL", (pt1[0], pt1[1] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

        if result.boxes is not None and result.boxes.id is not None:
            boxes = result.boxes.xyxy.cpu().numpy()
            ids_detectados = result.boxes.id.cpu().numpy().astype(int)
            clss = result.boxes.cls.cpu().numpy().astype(int)

            for box, id_obj, cls in zip(boxes, ids_detectados, clss):
                if cls in Classe:
                    x1, y1, x2, y2 = box

                    cx = int((x1 + x2) / 2)
                    cy = int((y1 + y2) / 2)
                    ponto_atual = (cx, cy)

                    # Usa o nome mapeado dentro do seu arquivo best(1).pt
                    label = f"{model.names[cls]} ID: {id_obj}"
                    cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
                    cv2.circle(frame, ponto_atual, 5, (0, 0, 255), -1)
                    cv2.putText(frame, label, (int(x1), int(y1) - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

                    if id_obj in historico:
                        ponto_anterior = historico[id_obj]

                        if intercepta_linha(pt1, pt2, ponto_anterior, ponto_atual):
                            if id_obj not in veiculos_contados:
                                veiculos_contados.add(id_obj)
                                vrum += 1

                    historico[id_obj] = ponto_atual

        cv2.rectangle(frame, (15, 15), (260, 65), (0, 0, 0), -1)
        cv2.putText(frame, f"CONTAGEM: {vrum}", (25, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

        out.write(frame)

    out.release()
    print(f"✅ Concluído: {video_in} | Total Contado: {vrum}\n")

LINHA_V1 = [(349, 386), (979, 388)]
LINHA_V2 = [(100, 218), (411, 209)]
LINHA_V3 = [(518, 238), (1260, 333)]
LINHA_V4 = [(271, 551), (727, 545)]

processar_video(Cam_v1, OUT_v1, LINHA_V1)
processar_video(Cam_v2, OUT_v2, LINHA_V2)
processar_video(Cam_v3, OUT_v3, LINHA_V3)
processar_video(Cam_v4, OUT_v4, LINHA_V4)

In [ ]:
!ffmpeg -i {OUT_v1} -c:v libx264 -c:a aac -pix_fmt yuv420p -y resultado_v1_web.mp4

In [ ]:
Video("resultado_v1_web.mp4", embed=True, width=1000)

In [ ]:
!ffmpeg -i {OUT_v2} -c:v libx264 -c:a aac -pix_fmt yuv420p -y resultado_v2_web.mp4

In [ ]:
Video("resultado_v2_web.mp4", embed=True, width=1000)

In [ ]:
!ffmpeg -i {OUT_v3} -c:v libx264 -c:a aac -pix_fmt yuv420p -y resultado_v3_web.mp4

In [ ]:
Video("resultado_v3_web.mp4", embed=True, width=1000)

In [ ]:
!ffmpeg -i {OUT_v4} -c:v libx264 -c:a aac -pix_fmt yuv420p -y resultado_v4_web.mp4

In [ ]:
Video("resultado_v4_web.mp4", embed=True, width=1000)

####Conclusão

Os resultados apresentaram bastante instabilidade. Talvez fossem melhores com mais epocas de treino, no entento, isso seria custoso (30 epocas deram +-2 horas).

As imagens do dataset também não são as melhores, há recomendações maiores para o bdd100k, mas, até onde vimos, são quase 10 GB de imagens e não é nativo do yolo.